# Part 4: Model Training and Callbacks

In this module, we will train the model using our dynamically augmented dataset. 
We utilize Keras callbacks to ensure the model trains efficiently and automatically saves the best weights.

In [ ]:
import os
import cv2
import shutil
import random
import datetime
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Define the dynamic export path for the Keras model with a timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"odia_ocr_cnn_{timestamp}.keras"

# Main Google Drive export path
drive_dir = "/content/drive/MyDrive/lipy/models"
os.makedirs(drive_dir, exist_ok=True)
checkpoint_path = os.path.join(drive_dir, filename)

# Define Keras Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
]

In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    class_weight=class_weight_dict
)
print("Training Complete. Best model saved to Google Drive:", checkpoint_path)

In [ ]:
# Attempt to automatically copy the resulting model directly into the local backend
# This works if the notebook is running locally or has access to the local filesystem structure
local_backend_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "backend", "models"))

if os.path.exists(local_backend_dir) and os.path.exists(checkpoint_path):
    local_model_path = os.path.join(local_backend_dir, filename)
    shutil.copy(checkpoint_path, local_model_path)
    print(f"\nSuccess! Copied model to local backend: {local_model_path}")
else:
    print("\nCould not automatically locate the local backend/models/ directory from this environment.")
    print("You may need to manually copy the .keras file into backend/models/.")